# Self-Attention を、自分の文で見る

図解では文が 1 つに固定されています（実測データを焼き込んでいるため）。
ここでは **あなたが書いた文** を GPT-2 small に通し、
「どの語がどの語をどれだけ参照したか」を自分で出します。

最後に、図解が読んでいるデータそのものを作り直して、値が合うことを確かめます。

- **戻る**: [Self-Attention のしくみ](https://manga-epoch.pages.dev/pub/epoch/arc2/figures_attention.html) — 同じ内容をスライダで動かせます
- **必要なもの**: Colab の標準環境のみ（GPU 不要・数分で終わります）
- 上から順に実行してください（Colab では `Shift + Enter`）。

## 0. 準備

GPT-2 small（パラメータ 1 億 2400 万）を読み込みます。初回はダウンロードに 1 分ほどかかります。

`attn_implementation="eager"` を指定しているのは、**注意の重みを取り出すため**です。
既定の高速実装は重みを内部で捨ててしまい、`output_attentions=True` が効きません。

In [ ]:
%pip install -q transformers

In [ ]:
import torch, numpy as np
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

tok = GPT2TokenizerFast.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2", attn_implementation="eager").eval()

D, H = model.config.n_embd, model.config.n_head       # 768 次元 / 12 ヘッド
HEAD_DIM = D // H                                      # 1 ヘッドあたり 64 次元
print(f"GPT-2 small: {D} 次元 / {H} ヘッド / 1 ヘッド {HEAD_DIM} 次元 / "
      f"ブロック {model.config.n_layer} 段 / 語彙 {model.config.vocab_size:,}")

## 1. 文をトークンに分ける

GPT-2 は単語ではなく **トークン** を扱います。よく出る綴りはひとかたまり、珍しい綴りは
分割されます。`Ġ` は「直前に空白がある」印です。

**ここを書き換えて、自分の文で試してください。** 英語のモデルなので英文が向いています。

In [ ]:
PROMPT = "Data visualization empowers users to"     # ← 自由に書き換えてください

ids = tok(PROMPT, return_tensors="pt")
tokens = tok.convert_ids_to_tokens(ids["input_ids"][0])
N = len(tokens)
for i, (t, v) in enumerate(zip(tokens, ids["input_ids"][0].tolist())):
    print(f"  {i}: {t!r:16} id={v}")
print(f"\n{N} トークン")

既定の文では `empowers` が `Ġem` と `powers` の 2 つに割れます。
「1 語 = 1 トークン」ではないことが、ここで見えます。

## 2. Q・K・V を取り出す

各トークンのベクトル $x$ から、3 つのベクトルを作ります。

$$Q = xW_Q,\quad K = xW_K,\quad V = xW_V$$

検索にたとえると、**Q = 探している条件**、**K = 各語が掲げている見出し**、
**V = その語が実際に渡す中身** です。

GPT-2 は 3 つの重みを 1 枚（`c_attn`）にまとめて持っていて、出力を 3 等分して使います。

In [ ]:
block = model.transformer.h[0]        # 第 1 ブロック

with torch.no_grad():
    h = model.transformer.wte(ids["input_ids"]) + model.transformer.wpe(
        torch.arange(N).unsqueeze(0))
    x = block.ln_1(h)                 # attention の直前は層正規化を通る
    qkv = block.attn.c_attn(x)        # (1, N, 3*768) — Q/K/V が横に並んでいる
    q, k, v = qkv.split(D, dim=2)

def to_heads(t):                      # (1, N, 768) -> (12, N, 64)
    return t.view(1, N, H, HEAD_DIM).permute(0, 2, 1, 3)[0]

qh, kh, vh = to_heads(q), to_heads(k), to_heads(v)
print("x   :", tuple(x.shape[1:]), "… 各トークンが 768 次元")
print("Q/K/V:", tuple(q.shape[1:]), "… それぞれ 768 次元")
print("ヘッドに割ると:", tuple(qh.shape), f"… {H} ヘッド × {N} トークン × {HEAD_DIM} 次元")

## 3. 重みができるまでの 4 段

$$\mathrm{Attention}(Q,K,V)=\mathrm{softmax}\!\left(\frac{QK^{\top}}{\sqrt{d_k}}+M\right)V$$

段ごとに分けると、こうなります。

1. **点数**: $QK^{\top}$ — 条件と見出しの内積。似ているほど大きい
2. **スケール**: $\div\sqrt{d_k}$ — 次元が大きいと内積も大きくなるので割って戻す
3. **マスク**: $+M$ — 自分より後ろの語を $-\infty$ にする（次の語を当てる訓練で、答えを見せないため）
4. **softmax**: 行ごとに合計 1 の割合へ

下のセルを実行すると、4 段を **手で** 組んだ結果が出ます。

In [ ]:
HEAD = 5          # ← 1〜12。ヘッドごとに見ているものが違います

hi = HEAD - 1
scores = qh[hi] @ kh[hi].T                        # ① 点数
scaled = scores / (HEAD_DIM ** 0.5)               # ② スケール
mask = torch.triu(torch.full((N, N), float("-inf")), diagonal=1)
masked = scaled + mask                            # ③ マスク
weights = torch.softmax(masked, dim=-1)           # ④ softmax

import pandas as pd
def show(m, name):
    df = pd.DataFrame(m.detach().numpy(), index=tokens, columns=tokens)
    print(f"\n【{name}】")
    print(df.round(3).to_string())

show(scores, "① 点数 QKᵀ")
show(scaled, "② ÷√d_k")
show(masked, "③ マスク後（-inf は後ろの語）")
show(weights, "④ softmax 後（各行の合計が 1）")
print("\n各行の合計:", weights.sum(dim=-1).detach().numpy().round(6))

### 検算 — 手で組んだものは、モデル自身の答えと一致するか

上の 4 段は「そう習ったから」ではなく、**モデルが実際に使っている計算**でなければ
意味がありません。モデルに attention を出させて突き合わせます。

In [ ]:
with torch.no_grad():
    out = model(**ids, output_attentions=True)

theirs = out.attentions[0][0, hi]                 # 第 1 ブロック・同じヘッド
err = (weights - theirs).abs().max().item()
print(f"手計算とモデル出力の最大差: {err:.2e}")
assert err < 1e-5, "一致しませんでした"
print("→ 一致。上の 4 段は、モデルが実際に通っている道筋です。")

## 4. ヘッドごとに、見ているものが違う

12 ヘッドを並べます。**濃いマスほど強く参照している**という意味です。
対角より右上が白いのはマスクの効果で、これはどのヘッドでも同じです。

In [ ]:
import matplotlib.pyplot as plt

A = out.attentions[0][0].detach().numpy()          # (12, N, N)
fig, axes = plt.subplots(3, 4, figsize=(15, 11))
for i, ax in enumerate(axes.flat):
    ax.imshow(A[i], cmap="Blues", vmin=0, vmax=1)
    ax.set_title(f"head {i+1}", fontsize=10)
    ax.set_xticks(range(N)); ax.set_yticks(range(N))
    # 日本語フォントが無い環境でも崩れないよう、ラベルはトークンのまま出す
    ax.set_xticklabels(tokens, rotation=90, fontsize=7)
    ax.set_yticklabels(tokens, fontsize=7)
fig.suptitle(f"block 1 attention  /  {PROMPT!r}", fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# 各ヘッドが「いちばん強く見た組」を並べると、役割の違いが読めます
for i in range(H):
    a = A[i].copy()
    a[np.triu_indices(N, 1)] = -1                  # マスク済みの領域は除く
    np.fill_diagonal(a, -1)                        # 自分自身も除く
    if a.max() <= 0:
        continue
    r, c = np.unravel_index(a.argmax(), a.shape)
    print(f"  head {i+1:2}: {tokens[r]!r} → {tokens[c]!r}  ({a[r, c]:.3f})")

既定の文だと、多くのヘッドが **先頭のトークン** を指します。意外に見えますが、
これは既知の性質です。

softmax は行の合計を必ず 1 にするので、「今は特に見たいものが無い」というヘッドも、
どこかへ重みを置かないといけません。その捨て場所として先頭トークンが使われます
（**attention sink** と呼ばれます）。**ヘッドが仕事をしていない印**であって、
先頭語が重要という意味ではありません。

そのなかで head 5 は `powers` → `Ġem` を **0.9 以上**で指します。
GPT-2 が `empowers` を 2 つに割ってしまったので、後半が前半を強く参照して
1 語ぶんの意味を組み立て直している――と読めます。
自分の文を入れると、どのヘッドが働くかは変わります。

## 5. 次の 1 語

12 ブロックを通り抜けた後、最後のトークンのベクトルを語彙 50,257 個分の点数（logit）に
変え、softmax で確率にします。温度 $T$ は分布の尖り方を決めます。

$$p_i=\frac{\exp(z_i/T)}{\sum_j \exp(z_j/T)}$$

$T$ を小さくすると 1 位に集中し、大きくすると平らになります。

In [ ]:
T = 1.0            # ← 0.3 や 2.0 に変えて、分布の変わり方を見てください

logits = out.logits[0, -1]
probs = torch.softmax(logits / T, dim=-1)
top = torch.topk(probs, 10)
print(f"{PROMPT!r} の次に来る語（温度 T={T}）\n")
for p, i in zip(top.values.tolist(), top.indices.tolist()):
    print(f"  {tok.decode([i])!r:16} {p*100:5.1f}%   logit={logits[i]:.2f}")

## 6. 逆向き — 図解のデータを作り直す

[Self-Attention の図解](https://manga-epoch.pages.dev/pub/epoch/arc2/figures_attention.html) は、
`gpt2_attention.json` という実測データを読んで描いています。
そのファイルを、ここで作り直します。

図解に「本物の数値」と書いてあることの裏を、読者側から取れる、ということです。

In [ ]:
import json

def build_figure_data(prompt, block_no=1):
    ids = tok(prompt, return_tensors="pt")
    # 図解は読みやすさのためデコード済みの語形（先頭の空白入り）で持っている。
    # そのまま差し替えられるよう、ここでも同じ形にする（表示用の Ġ 付きとは別）
    toks = [tok.decode([i]) for i in ids["input_ids"][0].tolist()]
    n = len(toks)
    bi = block_no - 1
    blk = model.transformer.h[bi]
    with torch.no_grad():
        out = model(**ids, output_attentions=True, output_hidden_states=True)
        x = blk.ln_1(out.hidden_states[bi])
        q, k, _ = blk.attn.c_attn(x).split(D, dim=2)
    hq = q.view(1, n, H, HEAD_DIM).permute(0, 2, 1, 3)[0]
    hk = k.view(1, n, H, HEAD_DIM).permute(0, 2, 1, 3)[0]
    m = torch.triu(torch.full((n, n), float("-inf")), diagonal=1)
    heads = []
    for h in range(H):
        s = hq[h] @ hk[h].T
        sc = s / (HEAD_DIM ** 0.5)
        heads.append({
            "attn":        [[round(x, 4) for x in r] for r in s.tolist()],
            "attn_scaled": [[round(x, 4) for x in r] for r in sc.tolist()],
            "attn_masked": [[None if x == float("-inf") else round(x, 4) for x in r]
                            for r in (sc + m).tolist()],
            "attn_softmax": [[round(x, 4) for x in r]
                             for r in torch.softmax(sc + m, dim=-1).tolist()],
        })
    lg = out.logits[0, -1]
    top = torch.topk(lg, 12)
    return {
        "source": "このノートブックで GPT-2 small を実行して生成",
        "model": "GPT-2 small", "block": block_no, "prompt": prompt,
        "tokens": toks, "heads": heads,
        "next_candidates": [{"token": tok.decode([i]), "logit": round(v, 3)}
                            for v, i in zip(top.values.tolist(), top.indices.tolist())],
    }

data = build_figure_data(PROMPT)
with open("gpt2_attention.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False)
print("gpt2_attention.json を書き出しました（Colab の左のファイル欄から取り出せます）")
print("トークン:", data["tokens"])

### 同梱データと突き合わせる

既定の文（`Data visualization empowers users to`）のままなら、公開されている
データと一致するはずです。ダウンロードして比べます。

In [ ]:
import urllib.request

URL = ("https://manga-epoch.pages.dev/notebooks/"
       "data/gpt2_attention.json")
try:
    ref = json.loads(urllib.request.urlopen(URL, timeout=20).read().decode("utf-8"))
except Exception as e:
    ref = None
    print("取得できませんでした:", e)

if ref and ref["prompt"] == PROMPT:
    mine = build_figure_data(PROMPT)
    d = max(abs(a - b)
            for hm, hr in zip(mine["heads"], ref["heads"])
            for ra, rb in zip(hm["attn_softmax"], hr["attn_softmax"])
            for a, b in zip(ra, rb))
    print(f"トークン列: {'一致' if mine['tokens'] == ref['tokens'] else '不一致'}")
    print(f"softmax 後の重み: 最大差 {d:.2e}")
    print("→ 図解に出ている数値は、ここで再現できるものです。")
elif ref:
    print(f"文を変えているので比較はしません（同梱は {ref['prompt']!r}）。")

差が完全に 0 にならないのは、同梱データが小数点以下を丸めているためです（$10^{-4}$ 程度）。

## 次に

- **図解に戻る**: [Self-Attention のしくみ](https://manga-epoch.pages.dev/pub/epoch/arc2/figures_attention.html)
  / [Transformer](https://manga-epoch.pages.dev/pub/epoch/arc2/figures_transformer.html)
- **手で書く**: [nn_from_scratch.ipynb](https://colab.research.google.com/github/manga-epoch/viewer/blob/main/notebooks/nn_from_scratch.ipynb) — NumPy だけで
  ニューラルネットと誤差逆伝播を組みます

---

## 出典とライセンス

このノートブックは、マンガ **EPOCH — 時代の前夜** の④「書く」レイヤーです。
[EPOCH について](https://manga-epoch.pages.dev)

ここで使う **GPT-2 small** は OpenAI が Modified MIT License で公開しているモデルです（Software Copyright (c) 2019 OpenAI）。同ライセンスは「本ソフトウェアが生成した内容には著作権表示を含めなくてよい」と定め、GPT-2 を使って作ったものであることを明示するよう求めています。このノートブックが出す数値はすべて GPT-2 small の出力です。

本ノートブックのコードは自由に改変して使えます。